In [9]:
# Leitura dos arquivos e criação de tabela unica 

import pandas as pd
import re
import os
from datetime import datetime

# pasta onde estão os arquivos .xls
path_raiz = r"C:\Users\Pedro\OneDrive\Documentos\Guias\arquivos_do_dia"

# pasta onde o arquivo final será salvo
path_saida = r"C:\Users\Pedro\OneDrive\Documentos\Guias"

# data de hoje para usar no nome do arquivo depois
dia_hj = datetime.now().strftime("%d-%m-%Y")

arquivos_xls = [arq for arq in os.listdir(path_raiz) if arq.endswith(".xls")]

lista_df = []

for arq in arquivos_xls:
    caminho_arquivo = os.path.join(path_raiz, arq)
    df = pd.read_html(caminho_arquivo, header=None, decimal=',',  thousands='.' )[1]
    lista_df.append(df)

df_fim = pd.concat(lista_df, ignore_index=False)

In [10]:
 ## - Inicio do tratamento : 

# remove espaços no início e fim dos nomes das colunas
df_fim.columns = df_fim.columns.str.strip()

# Renomiando colunas 
renomear_colunas = {
    "Emissor": "Cnpj Emissor",
    "Razão Social": "Razão Social Emissor",
    "Prestador": "Cnpj Prestador",
    "Razão Social.1": "Razão Social Prestador",
    "D.A.M.": "DAM"
}
df_fim = df_fim.rename(columns=renomear_colunas)

# Remoção de dados invalalidos das colunas de data Vencimento e Pagamento 
valores_invalidos = ["00/00/0000", "//", ""]

df_fim["Vencimento"] = df_fim["Vencimento"].replace(valores_invalidos, pd.NA)
df_fim["Pagamento"] = df_fim["Pagamento"].replace(valores_invalidos, pd.NA)

# Convertendo para datetime (PT-BR)
df_fim["Vencimento"] = pd.to_datetime(df_fim["Vencimento"], format="%d/%m/%Y", errors="coerce")
df_fim["Pagamento"] = pd.to_datetime(df_fim["Pagamento"], format="%d/%m/%Y", errors="coerce")

# Colunas que devem ser tratadas como TEXTO
colunas_texto = [
    "Cnpj Emissor", "Razão Social Emissor", "Cnpj Prestador", "Razão Social Prestador", "DAM", "Tipo", "Cancelada"]

for col in colunas_texto:
    if col in df_fim.columns:
        df_fim[col] = df_fim[col].astype("string")


# Datas com formato brasileiro (equivalente ao locale pt-BR)
df_fim["Vencimento"] = pd.to_datetime(df_fim["Vencimento"], format="%d/%m/%Y", errors="coerce")
df_fim["Pagamento"] = pd.to_datetime(df_fim["Pagamento"], format="%d/%m/%Y", errors="coerce")

# Limpar Cnpj Emissor deixando apenas dígitos
if "Cnpj Emissor" in df_fim.columns:
    df_fim["Cnpj Emissor"] = (
        df_fim["Cnpj Emissor"]
        .astype("string")              # garante que é texto
        .str.replace(r"\D", "", regex=True)  # remove tudo que não for dígito
    )

# Remover Dam duplicada
df_fim = df_fim.drop_duplicates(subset=["DAM"], keep="first")

# Criar coluna concatenada "Razao_Cnpj Emissor"
df_fim["Razao_Cnpj Emissor"] = (
    df_fim["Cnpj Emissor"].fillna("").astype(str)
    + " - "
    + df_fim["Razão Social Emissor"].fillna("").astype(str)
)

# Lista de CNPJs bloqueados (mesma do código M)
lista_bloqueio = [
    "07835050000138","19411731000158","83533217000194","21004012000164",
    "26734407000136","08537986000145","97624678000187","11670499000160",
    "12216767000131","24532754000150","64916243000157","04753613000231",
    "63838133000151","71672571000110","85809284000114","72026924000178",
    "15997136000195","56373945000103","41592516000150","07019500000203",
    "47621331000102","37249433000195","11681189000141","15997136000195",
    "56373945000103","41592516000150"
]

# Aplicar filtro removendo linhas cujo CNPJ_Emissor_TRATADO está na lista
df_fim = df_fim[~df_fim["Cnpj Emissor"].isin(lista_bloqueio)]

# # 5) Criar coluna CNPJ_Emissor_TRATADO com zero-padding (11/14 dígitos)
# def tratar_cnpj(v):
#     if pd.isna(v):
#         return pd.NA

#     v = str(v).strip()
#     if v == "":
#         return pd.NA

#     # garante só dígitos (por segurança)
#     v = re.sub(r"\D", "", v)

#     if len(v) < 11:
#         return v.zfill(11)
#     elif len(v) < 14:
#         return v.zfill(14)
#     else:
#         return v

# df_fim["CNPJ_Emissor_TRATADO"] = df_fim["Cnpj Emissor"].apply(tratar_cnpj)


In [5]:
# Exportar como Parquet no mesmo caminho
caminho_parquet = r"C:\Users\Pedro\OneDrive\Documentos\Guias\Guias_Tratado_" + dia_hj + ".parquet"
df_fim.to_parquet(caminho_parquet, index=False)
